# AI Service — Kaggle Deployment

**Yêu cầu trước khi chạy:**
1. GPU T4 hoặc P100 đã bật
2. Internet đã bật (Settings → Internet → On)
3. Secret `AI_INTERNAL_SERVICE_TOKEN` đã tạo và attach
4. Kaggle Dataset riêng tư chứa thư mục `runtime-artifacts` đã được gắn bằng **Add Input**

## Giai đoạn A — Thiết lập môi trường

In [ ]:
!nvidia-smi

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess

os.chdir("/kaggle/working")
print("Current directory:", os.getcwd())

repo = Path("/kaggle/working/Lab-Portal")

# Xóa thư mục clone dở, nhưng không xóa repository hợp lệ
if repo.exists() and not (repo / ".git").exists():
    shutil.rmtree(repo)
    print("Đã xóa thư mục clone dở.")

if not repo.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "dev",
            "https://github.com/nvtrung18/Lab-Portal.git",
            str(repo),
        ],
        cwd="/kaggle/working",
        check=True,
    )
else:
    print("Repository đã tồn tại:", repo)
    subprocess.run(["git", "fetch", "origin", "dev"], cwd=repo, check=True)
    subprocess.run(["git", "checkout", "dev"], cwd=repo, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", "dev"], cwd=repo, check=True)

os.chdir("/kaggle/working/Lab-Portal/ai-service")
print("AI service:", os.getcwd())

### Khôi phục runtime-artifacts từ Kaggle Dataset riêng tư

1. Tạo một Kaggle Dataset **Private** từ thư mục `runtime-artifacts`.
2. Trong notebook chọn **Add Input** và gắn Dataset đó.
3. Không cần chia sẻ Google Drive hoặc đặt model ở chế độ công khai.

In [ ]:
from pathlib import Path

input_root = Path("/kaggle/input")
print("Các input đã gắn:")
for path in sorted(input_root.iterdir()):
    print(" -", path)

In [ ]:
from pathlib import Path
import shutil

# Nếu tự động tìm không đúng, nhập đường dẫn hiển thị ở cell trên.
ARTIFACT_DATASET_PATH = ""
input_root = Path("/kaggle/input")
target = Path("/kaggle/working/runtime-artifacts")

if target.exists():
    print("runtime-artifacts đã tồn tại, bỏ qua tải lại.")
else:
    if ARTIFACT_DATASET_PATH:
        source = Path(ARTIFACT_DATASET_PATH)
        matches = [source]
    else:
        possible_roots = list(input_root.iterdir()) + list(input_root.rglob("runtime-artifacts"))
        matches = [
            path for path in possible_roots
            if (path / "shared-base").is_dir() and (path / "research-assistant").is_dir()
        ]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Cần đúng một runtime-artifacts trong /kaggle/input, tìm thấy: {matches}"
        )
    shutil.copytree(matches[0], target)
    print("\n✅ Đã copy runtime-artifacts từ:", matches[0])

# Liệt kê nội dung
for item in sorted(target.rglob("*")):
    if item.is_file():
        size_mb = item.stat().st_size / (1024 * 1024)
        print(f"  {item.relative_to(target)}  ({size_mb:.1f} MB)")

In [ ]:
from pathlib import Path

artifact_root = Path("/kaggle/working/runtime-artifacts")

revision = "cdbee75f17c01a7cc42f958dc650907174af0554"

base_path = artifact_root / f"shared-base/{revision}"
adapter_path = artifact_root / "research-assistant/1.0.0"

required_files = [
    base_path / "config.json",
    base_path / "model-00001-of-00003.safetensors",
    base_path / "model-00002-of-00003.safetensors",
    base_path / "model-00003-of-00003.safetensors",
    base_path / "model.safetensors.index.json",
    base_path / "tokenizer.json",
    adapter_path / "adapter_config.json",
    adapter_path / "adapter_model.safetensors",
]

missing = [str(path) for path in required_files if not path.is_file()]

print("Artifact root tồn tại:", artifact_root.exists())
print("Base tồn tại:", base_path.exists())
print("Adapter tồn tại:", adapter_path.exists())

if missing:
    print("\n❌ Thiếu file:")
    for path in missing:
        print("  ", path)
else:
    print("\n✅ Các file model quan trọng đã đầy đủ")

In [ ]:
%cd /kaggle/working/Lab-Portal/ai-service

%pip install -q --no-cache-dir -r requirements-runtime-t4.txt

In [ ]:
%pip uninstall -y torchvision

⚠️ **Restart session ngay bây giờ** (Runtime → Restart Session), rồi chạy tiếp từ Giai đoạn B

## Giai đoạn B — Khởi động AI Service

In [ ]:
import torch
import numpy
import transformers
import bitsandbytes
import peft
import importlib.util

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__)
print("NumPy:", numpy.__version__)
print("Transformers:", transformers.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("PEFT:", peft.__version__)
print("torchvision:", importlib.util.find_spec("torchvision"))

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_tool_planner.py",
        "tests/test_lab_mvp.py",
        "-q",
    ],
    cwd="/kaggle/working/Lab-Portal/ai-service",
    check=True,
)
print("AI REGRESSION TESTS: PASS")

In [ ]:
import os

# === CẤU HÌNH TOKEN ===
# Cách 1: Dùng Kaggle Secrets (khuyên dùng)
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["AI_INTERNAL_SERVICE_TOKEN"] = secrets.get_secret("AI_INTERNAL_SERVICE_TOKEN")

# Cách 2: Nhập thủ công (comment cách 1, bỏ comment phần này)
# from getpass import getpass
# os.environ["AI_INTERNAL_SERVICE_TOKEN"] = getpass("Nhập AI internal token: ")

# === CẤU HÌNH PATHS ===
os.environ["AI_ENVIRONMENT"] = "local"
os.environ["AI_MODEL_ARTIFACTS_PATH"] = (
    "/kaggle/working/Lab-Portal/ai-service/"
    "config/model-artifacts.runtime-t4.json"
)
# Trỏ đến thư mục đã tải từ Google Drive
os.environ["AI_MODEL_ARTIFACT_ROOT"] = (
    "/kaggle/working/runtime-artifacts"
)
os.environ["AI_RUNTIME_LOAD_ENABLED"] = "true"
os.environ["AI_RUNTIME_DEVICE"] = "cuda:0"
os.environ["AI_REQUEST_TIMEOUT_SECONDS"] = "300"

print("Độ dài token:", len(os.environ["AI_INTERNAL_SERVICE_TOKEN"]))
print("Artifact root:", os.environ["AI_MODEL_ARTIFACT_ROOT"])

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

log_path = Path("/kaggle/working/ai-service.log")
ai_log = log_path.open("w")

ai_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "app.main:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000",
    ],
    cwd="/kaggle/working/Lab-Portal/ai-service",
    env=os.environ.copy(),
    stdout=ai_log,
    stderr=subprocess.STDOUT,
)

print("AI service PID:", ai_process.pid)

In [ ]:
import time
import requests
from pathlib import Path

ready_url = "http://127.0.0.1:8000/ready"

for attempt in range(90):
    if ai_process.poll() is not None:
        print("❌ AI service đã dừng. Exit code:", ai_process.poll())
        print(
            Path("/kaggle/working/ai-service.log")
            .read_text(errors="replace")[-15000:]
        )
        break

    try:
        response = requests.get(
            ready_url,
            headers={
                "X-Internal-Service-Token":
                    os.environ["AI_INTERNAL_SERVICE_TOKEN"],
                "X-Request-Id":
                    "kaggle-ready-check",
            },
            timeout=10,
        )

        body = response.json()
        print(attempt + 1, response.status_code, body)

        if response.status_code == 200 and body.get("ready") is True:
            print("✅ AI SERVICE: READY")
            break

        if body.get("modelStatus") == "ERROR":
            print("❌ MODEL LOAD: ERROR")
            print(
                Path("/kaggle/working/ai-service.log")
                .read_text(errors="replace")[-15000:]
            )
            break

    except requests.RequestException:
        print("Đang chờ model...", attempt + 1)

    time.sleep(5)
else:
    print("⏰ Hết thời gian chờ model")

## Giai đoạn C — Tạo URL công khai (Cloudflare Tunnel)

In [ ]:
from pathlib import Path

cloudflared_path = Path("/usr/local/bin/cloudflared")

if not cloudflared_path.exists():
    !wget -q \
      https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
      -O /usr/local/bin/cloudflared

    !chmod +x /usr/local/bin/cloudflared
    print("✅ Đã tải cloudflared")
else:
    print("cloudflared đã tồn tại")

!/usr/local/bin/cloudflared --version

In [ ]:
import subprocess
import time
import re
from pathlib import Path

tunnel_log_path = Path("/kaggle/working/cloudflared.log")
tunnel_log = tunnel_log_path.open("w")

tunnel_process = subprocess.Popen(
    [
        "/usr/local/bin/cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8000",
        "--no-autoupdate",
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

public_url = None

for attempt in range(60):
    if tunnel_process.poll() is not None:
        print("❌ Tunnel đã dừng:", tunnel_process.poll())
        break

    content = tunnel_log_path.read_text(errors="replace")

    match = re.search(
        r"https://[a-z0-9-]+\.trycloudflare\.com",
        content,
    )

    if match:
        public_url = match.group(0)
        break

    print("Đang tạo tunnel...", attempt + 1)
    time.sleep(2)

if public_url:
    print(f"\n🌐 AI public URL: {public_url}")
    print(f"\n📋 Dùng URL này để cấu hình backend của bạn")
else:
    print("❌ Không lấy được URL")
    print(tunnel_log_path.read_text(errors="replace")[-10000:])

In [ ]:
import requests
import os

response = requests.get(
    f"{public_url}/ready",
    headers={
        "X-Internal-Service-Token":
            os.environ["AI_INTERNAL_SERVICE_TOKEN"],
        "X-Request-Id":
            "public-ready-check",
    },
    timeout=30,
)

print("HTTP:", response.status_code)
print(response.json())

## Regression test — tạo ca và phạm vi Lab

Chạy cell này sau khi AI service đã READY. Cell kiểm tra planner không đổi yêu cầu tạo thành thao tác đọc và không thay một Lab ngoài quyền bằng Lab được cấp quyền.

In [ ]:
import os
import requests

planner_url = "http://127.0.0.1:8000/v1/assistants/tool-request"
headers = {
    "X-Internal-Service-Token": os.environ["AI_INTERNAL_SERVICE_TOKEN"],
    "X-Request-Id": "kaggle-lab-planner-regression",
}
candidates = [
    {
        "assistantKey": "LAB_ASSISTANT",
        "schemaVersion": "v1",
        "toolId": "lab.available.slots.read",
        "description": "List future available time slots for managed Lab AI Research Lab",
        "resource": {"resourceType": "LABORATORY", "resourceId": 1},
        "parentResource": None,
    },
    {
        "assistantKey": "LAB_ASSISTANT",
        "schemaVersion": "v1",
        "toolId": "lab.shift.create.draft",
        "description": "Create a confirmation preview for a new time slot in managed Lab AI Research Lab",
        "resource": {"resourceType": "LABORATORY", "resourceId": 1},
        "parentResource": None,
    },
]
cases = [
    ("Tạo ca tại AI Research Lab ngày 10/09/2026 từ 8 giờ đến 10 giờ.", "TOOL_REQUEST"),
    ("Mở ca mới tại AI Research Lab ngày 10/09/2026 từ 8 giờ đến 10 giờ.", "TOOL_REQUEST"),
    ("Tạo ca tại Lab Robotics Lab ngày 10/09/2026 từ 8 giờ đến 10 giờ.", "REFUSAL"),
]

for index, (prompt, expected) in enumerate(cases, start=1):
    request_headers = {**headers, "X-Request-Id": f"kaggle-lab-planner-{index}"}
    response = requests.post(
        planner_url,
        headers=request_headers,
        json={"input": prompt, "candidates": candidates},
        timeout=300,
    )
    body = response.json()
    actual = body.get("decision")
    print(f"[{index}] HTTP {response.status_code} — {actual}: {prompt}")
    assert response.status_code == 200, body
    assert actual == expected, body
    if expected == "TOOL_REQUEST":
        assert body["toolRequest"]["toolId"] == "lab.shift.create.draft", body

print("KAGGLE LAB PLANNER REGRESSION: PASS")

## Debug — Xem log (chạy khi cần)

In [ ]:
from pathlib import Path
print(Path("/kaggle/working/ai-service.log").read_text(errors="replace")[-10000:])